# Solve the dual one multiplier at a time

The SVM dual has one unknown per training point, each held in the box $[0, C]$. Change a
single $\alpha_i$ and keep the others fixed, and the dual objective is a parabola in that
one number, with a maximum you can write down. Sweeping through the points and doing this
over and over solves the whole problem. Here you build that solver for a linear SVM,
measure how close it gets with the duality gap, and check the optimality conditions
point by point.

To keep each step to one variable, the bias is folded into the weights: every point gets
an extra feature equal to 1, and its weight plays the part of $b$. That weight is now
penalized with the others, so this is a slightly different model from `SVC`, and the
constraint $\sum_k \alpha_k y_k = 0$, which came from the free bias, disappears. The SVM is

$$
\min_{\mathbf{w}}\ P(\mathbf{w}) = \tfrac12\lVert\mathbf{w}\rVert^2 + C\sum_k \max(0,\ 1 - m_k),
\qquad m_k = y_k\,(\mathbf{w}\cdot\mathbf{x}_k),
$$

and its dual has no equality constraint:

$$
\max_{0 \le \alpha_k \le C}\ D(\boldsymbol\alpha) = \sum_k \alpha_k - \tfrac12\lVert\mathbf{w}(\boldsymbol\alpha)\rVert^2,
\qquad \mathbf{w}(\boldsymbol\alpha) = \sum_k \alpha_k y_k \mathbf{x}_k .
$$

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
neg = rng.normal([-1.0, -0.5], 1.0, size=(30, 2))
pos = rng.normal([1.0, 0.8], 1.0, size=(30, 2))
X = np.hstack([np.vstack([neg, pos]), np.ones((60, 1))])   # third column: the bias feature
y = np.r_[-np.ones(30), np.ones(30)]
C = 0.5
print(X.shape, "two classes that overlap, C =", C)

## Part 1 (coding task): The duality gap

Write `duality_gap(alpha, X, y, C)`. It forms $\mathbf{w}(\boldsymbol\alpha)$ and returns
$P(\mathbf{w}(\boldsymbol\alpha)) - D(\boldsymbol\alpha)$. For any $\boldsymbol\alpha$
in the box this is at least 0, and it is 0 exactly at the solution, so it measures how far
from optimal a candidate is without knowing the optimum.

In [ ]:
def duality_gap(alpha, X, y, C):
    alpha = np.asarray(alpha, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
X2 = np.array([[2.0, 1.0], [0.0, 1.0]])   # two points, bias column included
y2 = np.array([1.0, -1.0])
g0 = duality_gap(np.zeros(2), X2, y2, 2.0)
assert np.isclose(g0, 4.0), f"at alpha = 0, w = 0 and the gap is C times the number of points, 4.0, got {g0}"
g = duality_gap(np.array([0.5, 0.5]), X2, y2, 2.0)
assert np.isclose(g, 2.0), f"at alpha = (0.5, 0.5) with C = 2 the gap is 2.0, got {g}: build w from alpha * y, and sum the hinge losses rather than average them"
for seed in range(5):
    a = np.random.default_rng(seed).uniform(0, C, size=len(y))
    assert duality_gap(a, X, y, C) >= -1e-9, "the gap came out negative for an alpha in the box: check the signs of P and D"
print("Part 1 passed")

## Part 2 (coding task): Coordinate ascent

Write `coordinate_ascent(X, y, C, epochs)`. Start from $\boldsymbol\alpha = \mathbf{0}$ and
$\mathbf{w} = \mathbf{0}$. In each epoch, visit $i = 0, 1, \dots, n - 1$ in order and replace
$\alpha_i$ by the value in $[0, C]$ that maximizes $D$ with every other multiplier fixed.
Keep `w` equal to $\mathbf{w}(\boldsymbol\alpha)$ as you go, by adding the change in
$\alpha_i$ times $y_i\mathbf{x}_i$, so each step costs one dot product. Return
`alpha` and `w`.

To find the best value, work out on paper how $D$ changes when $\alpha_i$ becomes
$\alpha_i + \delta$: in terms of $\delta$, the margin $m_i$ and $\lVert\mathbf{x}_i\rVert^2$,
it is a downward parabola.

In [ ]:
def coordinate_ascent(X, y, C, epochs):
    n, d = X.shape
    alpha = np.zeros(n)
    w = np.zeros(d)
    # TODO
    raise NotImplementedError

In [ ]:
alpha, w = coordinate_ascent(X, y, C, epochs=100)
assert np.all(alpha >= -1e-12) and np.all(alpha <= C + 1e-12), f"alpha left the box [0, C]: min {alpha.min():.3f}, max {alpha.max():.3f}"
assert np.allclose(w, (alpha * y) @ X), "w no longer matches (alpha * y) @ X: update w by the change actually made to alpha_i"
gap = duality_gap(alpha, X, y, C)
assert gap < 1e-8, f"after 100 epochs the duality gap is {gap:.2e}, not below 1e-8: the step along alpha_i should be the top of the parabola, slope / ||x_i||^2, then clipped"
print(f"w = {np.round(w, 4)}, duality gap {gap:.1e}")
print("Part 2 passed")

## Part 3 (coding task): The optimality conditions, point by point

At the solution each point's multiplier and its margin $m_i$ must agree, in the way the
lesson's table of $\alpha_i$ values describes. Write `kkt_violation(alpha, X, y, C, tol=1e-6)`.
Compute $\mathbf{w}$ from `alpha`, then for every point find how far $m_i$ is from the
values its $\alpha_i$ allows (0 when it is within them), and return the largest of those
distances. Treat an $\alpha_i$ within `tol` of 0 or of $C$ as equal to it. For example, a
point with $\alpha_i = 0$ and $m_i = 0.7$ is off by 0.3.

In [ ]:
def kkt_violation(alpha, X, y, C, tol=1e-6):
    alpha = np.asarray(alpha, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
v0 = kkt_violation(np.zeros(len(y)), X, y, C)
assert np.isclose(v0, 1.0), f"at alpha = 0, w = 0, every margin is 0 and the violation is 1.0, got {v0}"
X2 = np.array([[2.0, 1.0], [0.0, 1.0]])   # the two points from Part 1
v2 = kkt_violation(np.array([0.25, 0.5]), X2, np.array([1.0, -1.0]), 2.0)
assert np.isclose(v2, 0.75), f"with alpha = (0.25, 0.5) on the two points of Part 1 the margins are 0.75 and 0.25, so the violation is 0.75, got {v2}: return the largest, not an average"
v = kkt_violation(alpha, X, y, C)
assert v < 1e-6, f"the solution from Part 2 should meet every condition, but the largest violation is {v:.3f}"
nudged = alpha.copy()
free = np.flatnonzero((alpha > 1e-6) & (alpha < C - 1e-6))
nudged[free[0]] += 0.05
assert kkt_violation(nudged, X, y, C) > 1e-3, "moving one free multiplier by 0.05 should break the conditions"
sv = alpha > 1e-6
alpha_sv, w_sv = coordinate_ascent(X[sv], y[sv], C, epochs=100)
print(f"{sv.sum()} support vectors: {len(free)} on the margin, {(alpha >= C - 1e-6).sum()} at alpha = C")
print(f"refit on the support vectors only: largest change in w {np.abs(w_sv - w).max():.1e}")
print("Part 3 passed")

## Solution

### Part 1: The duality gap

In [ ]:
def duality_gap(alpha, X, y, C):
    alpha = np.asarray(alpha, dtype=float)
    w = (alpha * y) @ X
    primal = 0.5 * w @ w + C * np.maximum(0.0, 1.0 - y * (X @ w)).sum()
    dual = alpha.sum() - 0.5 * w @ w
    return primal - dual

**Why it works.** For $\boldsymbol\alpha$ in the box, every candidate $\mathbf{w}$ has
$P(\mathbf{w}) \ge D(\boldsymbol\alpha)$ (weak duality), and for an SVM the two optima are
equal (strong duality). So the gap of $\mathbf{w}(\boldsymbol\alpha)$ is an upper bound on
how far both $P$ and $D$ are from their common optimum. On the two-point check,
$\mathbf{w} = (1, 0)$, $P = 0.5 + 2(0 + 1) = 2.5$ and $D = 1 - 0.5 = 0.5$.

### Part 2: Coordinate ascent

In [ ]:
def coordinate_ascent(X, y, C, epochs):
    n, d = X.shape
    alpha = np.zeros(n)
    w = np.zeros(d)
    sq = (X ** 2).sum(axis=1)                    # ||x_i||^2, the curvature of each parabola
    for _ in range(epochs):
        for i in range(n):
            slope = 1.0 - y[i] * (w @ X[i])          # dD/d(alpha_i) at the current point
            new = min(max(alpha[i] + slope / sq[i], 0.0), C)
            w += (new - alpha[i]) * y[i] * X[i]
            alpha[i] = new
    return alpha, w

**Why it works.** With $\delta$ added to $\alpha_i$, $\mathbf{w}$ moves by
$\delta y_i\mathbf{x}_i$ and

$$
D(\boldsymbol\alpha + \delta\mathbf{e}_i) = D(\boldsymbol\alpha) + \delta\,(1 - m_i) - \tfrac12\delta^2\lVert\mathbf{x}_i\rVert^2 .
$$

Its top is at $\delta = (1 - m_i)/\lVert\mathbf{x}_i\rVert^2$.
A parabola that opens downward and is cut to an interval has its best point at the top
when the top is inside, and otherwise at the nearer end, so clipping
$\alpha_i + \delta$ to $[0, C]$ gives the exact one-variable maximum. Each step can only
raise $D$, and on this data 100 sweeps close the gap to rounding error.

### Part 3: The optimality conditions, point by point

In [ ]:
def kkt_violation(alpha, X, y, C, tol=1e-6):
    alpha = np.asarray(alpha, dtype=float)
    m = y * (X @ ((alpha * y) @ X))
    at_zero = alpha <= tol
    at_c = alpha >= C - tol
    free = ~at_zero & ~at_c
    worst = np.zeros_like(m)
    worst[at_zero] = np.maximum(0.0, 1.0 - m[at_zero])
    worst[at_c] = np.maximum(0.0, m[at_c] - 1.0)
    worst[free] = np.abs(m[free] - 1.0)
    return worst.max()

**Why it works.** The conditions are complementary slackness for the two constraints of each
point. $\alpha_i > 0$ forces the margin constraint to be tight or violated,
$m_i = 1 - \xi_i \le 1$. $\alpha_i < C$ means the slack's own multiplier $C - \alpha_i$ is
positive, which forces $\xi_i = 0$, so $m_i \ge 1$. Both together give $m_i = 1$. The last
lines refit on the 15 support vectors alone and get the same $\mathbf{w}$: the 45 points with
$\alpha_i = 0$ never mattered.